# Train carriage anomaly data generator — doors & bogies

A parameterised synthetic-data generator for **autoencoder-based anomaly detection**
on rolling-stock condition data. Everything is derived from physics-flavoured raw
waveforms, so the per-event feature tables and the raw traces are mutually consistent.

**What it produces**

| Output | Grain | Use |
|---|---|---|
| `door_events` | one row per door open/close cycle | dense feature autoencoder |
| `bogie_windows` | one row per 2 s ride-monitoring snapshot | dense feature autoencoder |
| `door_raw` | 3 channels × 9 s @ 200 Hz (sampled subset) | Conv1D / LSTM sequence autoencoder |
| `bogie_raw` | 2 channels × 2 s @ 5 kHz (sampled subset) | Conv1D / LSTM sequence autoencoder |

**Ground truth** — every row carries `fault_type`, `severity` (0→1), `label`,
`onset_mode` (gradual vs abrupt) and `dq_flag` (data-quality fault). Use labels for
*evaluation only*; train the autoencoder on the healthy commissioning window.

**Seeded faults**

- *Door*: obstruction, actuator degradation, misalignment, intermittent lock sensor
- *Bogie*: wheel flat, bearing outer-race (BPFO) defect, hot axlebox, damper degradation, hunting instability
- *Sensor / data quality*: stuck-at, dropout (NaN), clipping, calibration drift

Set `CFG.gradual_fraction = 1.0` for a pure early-warning (slow-drift-only) dataset.

In [ ]:
# ── Cell 1: setup & configuration ──────────────────────────────────────────────
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

OUT_DIR = "train_ae_data"
os.makedirs(OUT_DIR, exist_ok=True)


@dataclass
class Config:
    # reproducibility
    seed: int = 20260831

    # fleet
    n_carriages: int = 8
    doors_per_carriage: int = 4
    bogies_per_carriage: int = 2

    # service horizon
    start_date: str = "2026-04-01"
    n_days: int = 60
    stops_per_day: int = 16          # door cycles per door per weekday
    runs_per_day: int = 6            # bogie monitoring snapshots per bogie per weekday

    # raw sampling
    door_fs: int = 200               # Hz
    door_window_s: float = 9.0
    bogie_fs: int = 5000             # Hz
    bogie_window_s: float = 2.0

    # how much raw to keep on disk (faulty events kept at ~6x this rate — stratified)
    raw_keep_fraction_door: float = 0.04
    raw_keep_fraction_bogie: float = 0.08

    # fault population
    healthy_warmup_days: int = 21    # guaranteed-clean commissioning window (AE training set)
    p_door_fault: float = 0.30       # share of doors that develop a fault
    p_bogie_fault: float = 0.30      # share of bogies that develop a fault
    gradual_fraction: float = 0.70   # of faulty assets, share with slow drift (vs abrupt step)
    p_sensor_fault: float = 0.12     # share of assets with a data-quality episode
    obstruction_rate: float = 0.004  # per-cycle obstruction probability on a healthy door

    # environment
    temp_mean_c: float = 17.0
    temp_seasonal_c: float = 8.0
    temp_diurnal_c: float = 5.0


CFG = Config()
rng = np.random.default_rng(CFG.seed)

DOOR_FAULTS = ["obstruction_prone", "actuator_degradation", "misalignment", "lock_sensor_intermittent"]
BOGIE_FAULTS = ["wheel_flat", "bearing_bpfo", "hot_axlebox", "damper_degradation", "hunting_instability"]
SENSOR_FAULTS = ["stuck_at", "dropout", "clipping", "calibration_drift"]

DATES = pd.date_range(CFG.start_date, periods=CFG.n_days, freq="D")
print(f"Horizon {DATES[0].date()} → {DATES[-1].date()}  ({CFG.n_days} days)")
print(f"Fleet: {CFG.n_carriages} carriages, "
      f"{CFG.n_carriages * CFG.doors_per_carriage} doors, "
      f"{CFG.n_carriages * CFG.bogies_per_carriage} bogies")

## Cell 2 — asset registry, per-asset baselines and the fault schedule

Each door and bogie gets its own nominal behaviour (friction, travel times, wheel
diameter, track exposure). That per-asset spread is what makes the problem realistic:
a healthy door on carriage 3 does not look identical to a healthy door on carriage 7,
so the autoencoder has to learn a *manifold*, not a point.

Faults are scheduled per asset with an onset day and a ramp. `severity(day)` is the
single knob every simulator reads.

In [ ]:
# ── Cell 2: assets & fault schedule ────────────────────────────────────────────
def assign_faults(assets_df, fault_pool, p_fault, cfg, rng):
    """Pick an exact number of faulty assets and spread the fault types evenly, so a
    template dataset always contains at least one example of every fault type."""
    n = len(assets_df)
    k = min(n, max(len(fault_pool), int(round(p_fault * n))))
    victims = rng.choice(assets_df.index.to_numpy(), size=k, replace=False)

    pool = list(fault_pool)
    rng.shuffle(pool)
    types = [pool[j % len(pool)] for j in range(k)]

    assets_df = assets_df.copy()
    assets_df["fault_type"] = "none"
    assets_df["onset_day"] = np.inf
    assets_df["ramp_days"] = 1.0
    assets_df["max_severity"] = 0.0
    assets_df["onset_mode"] = "none"

    for idx, ft in zip(victims, types):
        gradual = rng.random() < cfg.gradual_fraction
        onset = float(rng.integers(cfg.healthy_warmup_days,
                                   max(cfg.healthy_warmup_days + 1, cfg.n_days - 4)))
        if gradual:
            ramp = max(0.35 * (cfg.n_days - onset), 3.0) * float(rng.uniform(0.7, 1.6))
            mode = "gradual"
        else:
            ramp = float(rng.uniform(0.2, 1.5))
            mode = "abrupt"
        assets_df.loc[idx, ["fault_type", "onset_day", "ramp_days", "max_severity", "onset_mode"]] = [
            ft, onset, max(ramp, 0.2), float(rng.uniform(0.55, 1.0)), mode
        ]
    return assets_df


def severity_at(day, onset, ramp, max_sev):
    """Sigmoid-ish severity ramp in [0, max_sev]; 0 before onset."""
    if day < onset:
        return 0.0
    x = min(1.0, (day - onset) / ramp)
    return float(max_sev * (x * x * (3.0 - 2.0 * x)))   # smoothstep


def build_assets(cfg, rng):
    doors, bogies = [], []
    for c in range(1, cfg.n_carriages + 1):
        car = f"C{c:02d}"
        route = "R1" if c % 2 else "R2"
        for d in range(1, cfg.doors_per_carriage + 1):
            doors.append(dict(
                asset_id=f"{car}-D{d}", asset_type="door", carriage_id=car, position=d, route=route,
                friction=float(rng.normal(1.0, 0.07)),        # nominal drive friction multiplier
                t_open=float(rng.normal(1.75, 0.10)),         # s
                t_close=float(rng.normal(2.05, 0.12)),        # s
                gap_mm=float(rng.normal(6.0, 0.6)),           # leaf-to-frame gap when closed
                lock_ms=float(rng.normal(160.0, 18.0)),       # lock engage delay
                i_idle=float(rng.normal(0.36, 0.03)),         # A
                cycles0=float(rng.integers(40_000, 180_000)), # lifetime cycles at t=0
            ))
        for b in range(1, cfg.bogies_per_carriage + 1):
            bogies.append(dict(
                asset_id=f"{car}-B{b}", asset_type="bogie", carriage_id=car, position=b, route=route,
                wheel_dia_m=float(rng.normal(0.860, 0.006)),  # worn wheels differ slightly
                track_gain=float(rng.normal(1.0, 0.09)),      # ride-quality exposure
                bounce_g=float(rng.normal(0.055, 0.008)),     # body bounce amplitude
                sway_g=float(rng.normal(0.045, 0.007)),
                temp_offset_c=float(rng.normal(0.0, 1.4)),    # axlebox sensor offset
                n_balls=int(rng.integers(10, 13)),
            ))
    doors_df = assign_faults(pd.DataFrame(doors), DOOR_FAULTS, cfg.p_door_fault, cfg, rng)
    bogies_df = assign_faults(pd.DataFrame(bogies), BOGIE_FAULTS, cfg.p_bogie_fault, cfg, rng)
    return doors_df, bogies_df


doors_df, bogies_df = build_assets(CFG, rng)

print(doors_df.groupby("fault_type").size().rename("doors").to_string())
print()
print(bogies_df.groupby("fault_type").size().rename("bogies").to_string())


def ambient_c(day_idx, hour, cfg):
    """Seasonal + diurnal ambient temperature."""
    seasonal = cfg.temp_mean_c + cfg.temp_seasonal_c * math.sin(2 * math.pi * (day_idx + 20) / 365.0)
    diurnal = cfg.temp_diurnal_c * math.sin(2 * math.pi * (hour - 9) / 24.0)
    return seasonal + diurnal

## Cell 3 — door cycle: raw waveform simulator

Three channels at 200 Hz over a 9 s window: leaf **position** (0 = closed, 1 = open),
motor **current**, and the **lock** proximity switch.

Position is built from smoothstep segments, so velocity and acceleration are smooth
and the current follows from them: `i = i_idle + k_v·|v|·friction + k_a·|a| + inrush + rub + noise`.
An obstruction inserts a stall + reopen + second close attempt, which is what makes the
current trace bimodal and the cycle long.

In [ ]:
# ── Cell 3: door waveform ──────────────────────────────────────────────────────
_TRAPZ = np.trapezoid if hasattr(np, "trapezoid") else np.trapz   # numpy 2.0 renamed trapz


def _smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


def _profile(t, segments, p0=0.0):
    """segments = [(t_start, t_end, p_target), ...] in order; position holds between them."""
    p = np.full_like(t, p0)
    cur = p0
    for ts, te, pe in segments:
        frac = _smoothstep((t - ts) / max(te - ts, 1e-6))
        p = np.where(t >= ts, cur + (pe - cur) * frac, p)
        cur = pe
    return p


def simulate_door_cycle(door, sev, ambient, rng, cfg, force_obstruction=False):
    fs = cfg.door_fs
    n = int(round(cfg.door_window_s * fs))
    t = np.arange(n) / fs
    fault = door["fault_type"]

    # cold grease is stiffer
    fric = door["friction"] * (1.0 + 0.012 * max(0.0, 15.0 - ambient))
    t_open, t_close = door["t_open"], door["t_close"]
    gap_mm, lock_ms = door["gap_mm"], door["lock_ms"]
    rub = 0.0
    p_obstruct = cfg.obstruction_rate
    lock_dropout = False

    if fault == "actuator_degradation":
        fric *= 1.0 + 1.7 * sev
        t_open *= 1.0 + 0.26 * sev
        t_close *= 1.0 + 0.34 * sev
        lock_ms += 45.0 * sev
    elif fault == "misalignment":
        gap_mm += 9.5 * sev
        t_close *= 1.0 + 0.10 * sev
        rub = 2.2 * sev                       # leaf rubs the frame near the closed end
        lock_ms += 120.0 * sev
        p_obstruct += 0.05 * sev              # false obstruction detections
    elif fault == "lock_sensor_intermittent":
        if rng.random() < 0.10 + 0.45 * sev:
            lock_ms += float(rng.uniform(200, 1100)) * sev
        lock_dropout = rng.random() < 0.05 + 0.30 * sev
    elif fault == "obstruction_prone":
        p_obstruct += 0.35 * sev

    obstruct = force_obstruction or (rng.random() < p_obstruct)

    # ---- position waypoints ----
    t0 = 0.30
    segs = [(t0, t0 + t_open, 1.0)]
    t_dwell_end = t0 + t_open + 1.6
    stall_t0 = stall_t1 = None
    if obstruct:
        p_stall = float(rng.uniform(0.30, 0.55))
        s0 = t_dwell_end
        s1 = s0 + t_close * (1.0 - p_stall) * 0.9
        segs += [(s0, s1, p_stall)]
        stall_t0, stall_t1 = s1, s1 + 0.28
        segs += [(stall_t1, stall_t1 + 0.60, 0.92)]            # reopen
        r0 = stall_t1 + 0.60 + 0.45
        segs += [(r0, r0 + t_close, 0.0)]                      # second close attempt
        close_end = r0 + t_close
    else:
        segs += [(t_dwell_end, t_dwell_end + t_close, 0.0)]
        close_end = t_dwell_end + t_close

    p = np.clip(_profile(t, segs), 0.0, 1.0)
    v = np.gradient(p, 1.0 / fs)
    a = np.gradient(v, 1.0 / fs)
    moving = np.abs(v) > 2e-3

    # ---- motor current ----
    i = np.full(n, door["i_idle"])
    i += (5.8 * np.abs(v) * fric + 0.22 * np.abs(a)) * moving
    for ts in [s[0] for s in segs]:                            # inrush at each motion start
        i += 2.4 * np.exp(-np.maximum(t - ts, 0.0) / 0.055) * (t >= ts) * (t < ts + 0.5)
    if rub > 0:                                                # rubbing near the closed end
        i += rub * np.exp(-((p - 0.10) / 0.09) ** 2) * moving
    if obstruct:                                               # stall current
        i += 6.5 * ((t >= stall_t0) & (t <= stall_t1 + 0.05))
    i += rng.normal(0.0, 0.045, n)
    i = np.clip(i, 0.0, None)

    # ---- lock switch ----
    lock = np.zeros(n, dtype=np.int8)
    t_lock = close_end + lock_ms / 1000.0
    lock[(t >= t_lock)] = 1
    lock[t < t0] = 1                                           # locked before unlock command
    if lock_dropout:                                           # intermittent sensor
        k = int(rng.integers(1, 4))
        for _ in range(k):
            b = float(rng.uniform(t_lock, cfg.door_window_s - 0.2))
            lock[(t >= b) & (t < b + rng.uniform(0.05, 0.25))] = 0

    meta = dict(obstructed=int(obstruct), lock_ms_true=lock_ms, gap_mm=gap_mm,
                gap_mm_meas=float(gap_mm + rng.normal(0.0, 0.15)),
                lock_dropout=int(lock_dropout))
    return t, p, i, lock, meta


def extract_door_features(t, p, i, lock, meta, fs):
    """Everything here is measured off the raw traces — no leakage of hidden state."""
    dt = 1.0 / fs
    open_idx = int(np.argmax(p >= 0.95)) if (p >= 0.95).any() else len(p) - 1

    def _cross(sig, thr, lo, hi, rising=True):
        seg = sig[lo:hi]
        m = (seg >= thr) if rising else (seg <= thr)
        return (lo + int(np.argmax(m))) if m.any() else hi

    i_o0 = _cross(p, 0.05, 0, open_idx + 1, True)
    t_open_s = (open_idx - i_o0) * dt

    peak_idx = int(np.argmax(p))
    tail = p[peak_idx:]
    last_hi = peak_idx + int(np.max(np.nonzero(tail >= 0.95)[0])) if (tail >= 0.95).any() else peak_idx
    i_c1 = _cross(p, 0.05, last_hi, len(p), False)
    t_close_s = (i_c1 - last_hi) * dt

    v_tail = np.gradient(p, dt)[peak_idx:]
    reopening = v_tail > 0.02
    n_reopen = int(np.sum(np.diff(reopening.astype(int)) == 1))

    close_slice = slice(last_hi, min(i_c1 + 1, len(i)))
    open_slice = slice(i_o0, open_idx + 1)
    lock_on = np.nonzero(lock[peak_idx:])[0]
    if lock_on.size:
        lock_delay_ms = float((lock_on[0] + peak_idx - i_c1) * dt * 1000.0)
        incomplete = 0
    else:                                   # cycle never completed inside the capture window
        lock_delay_ms = float((len(p) - i_c1) * dt * 1000.0)
        incomplete = 1

    return dict(
        t_open_s=float(t_open_s),
        t_close_s=float(t_close_s),
        peak_i_open_a=float(i[open_slice].max() if open_slice.stop > open_slice.start else np.nan),
        peak_i_close_a=float(i[close_slice].max() if close_slice.stop > close_slice.start else np.nan),
        mean_i_close_a=float(i[close_slice].mean() if close_slice.stop > close_slice.start else np.nan),
        rms_i_close_a=float(np.sqrt(np.mean(i[close_slice] ** 2)) if close_slice.stop > close_slice.start else np.nan),
        charge_as=float(_TRAPZ(i, dx=dt)),
        max_di_dt_a_s=float(np.abs(np.gradient(i, dt)).max()),
        i_std_a=float(i.std()),
        n_reopen=n_reopen,
        lock_delay_ms=lock_delay_ms,
        lock_unstable=int((np.diff(lock.astype(int)) != 0).sum() > 2),
        cycle_incomplete=incomplete,
        door_gap_mm=float(meta["gap_mm_meas"]),
        peak_v_norm_s=float(np.abs(np.gradient(p, dt)).max()),
    )

## Cell 4 — bogie ride snapshot: raw waveform simulator

Two accelerometer channels at 5 kHz over 2 s (vertical + lateral), plus scalar
axlebox temperature and speed.

Baseline = pink track irregularity + sleeper-passing tone (v / 0.65 m) + car-body
bounce/sway + a small once-per-revolution unbalance. Faults ride on top:

- **wheel flat** — impulse train at the wheel rotation frequency `f_rot = v / (π·D)`, each impact ringing at ~350 Hz
- **bearing BPFO** — impulse train at `0.4 · n_balls · f_rot`, ringing at ~1.5 kHz, amplitude-modulated once per revolution (classic envelope-spectrum signature)
- **hot axlebox** — temperature rise with only a mild vibration change
- **damper degradation** — energy piles up in the 0.7–4 Hz body modes
- **hunting instability** — sustained ~4.5 Hz lateral oscillation that grows with speed

In [ ]:
# ── Cell 4: bogie waveform ─────────────────────────────────────────────────────
def _colored_noise(n, rng, beta=1.0):
    """Noise with a 1/f**(beta/2) amplitude spectrum (beta=1 → pink), unit std."""
    w = rng.normal(0.0, 1.0, n)
    W = np.fft.rfft(w)
    f = np.fft.rfftfreq(n, d=1.0)
    f[0] = f[1] if len(f) > 1 else 1.0
    W = W / (f ** (beta / 2.0))
    x = np.fft.irfft(W, n=n)
    s = x.std()
    return x / (s if s > 0 else 1.0)


def _impulse_train(n, fs, freq, amp, ring_hz, decay_s, rng, jitter=0.02, modulation=None, t=None):
    """Impulse train convolved with a decaying-sinusoid resonance ('ringing')."""
    if freq <= 0 or amp <= 0:
        return np.zeros(n)
    train = np.zeros(n)
    period = 1.0 / freq
    times = np.arange(rng.uniform(0, period), n / fs, period)
    idx = np.round((times * (1.0 + rng.normal(0, jitter, times.size))) * fs).astype(int)
    idx = idx[(idx >= 0) & (idx < n)]
    a = amp * (1.0 + rng.normal(0.0, 0.12, idx.size))
    if modulation is not None:
        a = a * modulation[idx]
    np.add.at(train, idx, a)
    kn = max(4, int(decay_s * 6 * fs))
    tk = np.arange(kn) / fs
    kernel = np.exp(-tk / decay_s) * np.sin(2 * np.pi * ring_hz * tk)
    return np.convolve(train, kernel)[:n]


def simulate_bogie_window(bogie, sev, speed_kmh, ambient, rng, cfg):
    fs = cfg.bogie_fs
    n = int(round(cfg.bogie_window_s * fs))
    t = np.arange(n) / fs
    fault = bogie["fault_type"]

    v = speed_kmh / 3.6
    d = bogie["wheel_dia_m"]
    f_rot = v / (np.pi * d)
    f_sleeper = v / 0.65
    f_bpfo = 0.40 * bogie["n_balls"] * f_rot
    gain = bogie["track_gain"] * (0.35 + 0.030 * speed_kmh)

    av = 0.09 * gain * _colored_noise(n, rng, beta=1.0)
    al = 0.07 * gain * _colored_noise(n, rng, beta=1.0)
    av += 0.05 * gain * np.sin(2 * np.pi * f_sleeper * t + rng.uniform(0, 2 * np.pi))
    av += bogie["bounce_g"] * np.sin(2 * np.pi * 1.3 * t + rng.uniform(0, 2 * np.pi))
    al += bogie["sway_g"] * np.sin(2 * np.pi * 0.85 * t + rng.uniform(0, 2 * np.pi))
    av += 0.018 * np.sin(2 * np.pi * f_rot * t + rng.uniform(0, 2 * np.pi))   # residual unbalance

    temp_rise = 11.0 + 0.30 * speed_kmh + rng.normal(0.0, 0.8)

    if fault == "wheel_flat":
        av += _impulse_train(n, fs, f_rot, 1.5 * sev * (0.4 + speed_kmh / 60.0),
                             ring_hz=350.0, decay_s=0.004, rng=rng)
        al += _impulse_train(n, fs, f_rot, 0.5 * sev * (0.4 + speed_kmh / 60.0),
                             ring_hz=350.0, decay_s=0.004, rng=rng)
        temp_rise += 1.5 * sev
    elif fault == "bearing_bpfo":
        mod = 1.0 + 0.55 * np.sin(2 * np.pi * f_rot * t)     # once-per-rev load modulation
        av += _impulse_train(n, fs, f_bpfo, 0.85 * sev, ring_hz=1500.0, decay_s=0.0016,
                             rng=rng, modulation=mod)
        temp_rise += 6.0 * sev
    elif fault == "hot_axlebox":
        temp_rise += 26.0 * sev
        av += 0.02 * sev * _colored_noise(n, rng, beta=0.5)
    elif fault == "damper_degradation":
        av += 0.32 * sev * np.sin(2 * np.pi * 1.15 * t + rng.uniform(0, 2 * np.pi))
        av += 0.18 * sev * np.sin(2 * np.pi * 2.6 * t + rng.uniform(0, 2 * np.pi))
        al += 0.22 * sev * np.sin(2 * np.pi * 1.9 * t + rng.uniform(0, 2 * np.pi))
        av += 0.05 * sev * _colored_noise(n, rng, beta=2.0)
    elif fault == "hunting_instability":
        amp = 0.55 * sev * (speed_kmh / 60.0) ** 2
        al += amp * np.sin(2 * np.pi * 4.6 * t + rng.uniform(0, 2 * np.pi))
        al += 0.25 * amp * np.sin(2 * np.pi * 9.2 * t + rng.uniform(0, 2 * np.pi))

    axlebox_c = ambient + temp_rise + bogie["temp_offset_c"]
    meta = dict(f_rot_hz=f_rot, f_bpfo_hz=f_bpfo, speed_kmh=speed_kmh,
                axlebox_temp_c=axlebox_c, ambient_c=ambient)
    return t, av.astype(np.float64), al.astype(np.float64), meta


# ---- signal-processing helpers (numpy only, no scipy needed) ----
def _kurtosis(x):
    x = np.asarray(x, float)
    s = x.std()
    return float(np.mean((x - x.mean()) ** 4) / (s ** 4)) if s > 0 else np.nan


def _band_rms(mag, freqs, lo, hi):
    m = (freqs >= lo) & (freqs < hi)
    return float(np.sqrt(np.sum(mag[m] ** 2)) if m.any() else 0.0)


def _bandpass(x, fs, lo, hi):
    n = x.size
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, d=1.0 / fs)
    X[(f < lo) | (f > hi)] = 0.0
    return np.fft.irfft(X, n=n)


def _envelope(x):
    n = x.size
    X = np.fft.fft(x)
    h = np.zeros(n)
    h[0] = 1.0
    if n % 2 == 0:
        h[n // 2] = 1.0
        h[1:n // 2] = 2.0
    else:
        h[1:(n + 1) // 2] = 2.0
    return np.abs(np.fft.ifft(X * h))


def extract_bogie_features(t, av, al, meta, fs):
    n = av.size
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    mag_v = np.abs(np.fft.rfft(av)) * 2.0 / n
    mag_l = np.abs(np.fft.rfft(al)) * 2.0 / n

    # Envelope analysis, the standard rolling-stock recipe: band-pass around the
    # structural resonance the defect excites, then look at the envelope spectrum.
    ef = np.fft.rfftfreq(n, d=1.0 / fs)
    env_lo = _envelope(_bandpass(av, fs, 150.0, 600.0))     # wheel-flat impact ring (~350 Hz)
    env_hi = _envelope(_bandpass(av, fs, 800.0, 2400.0))    # bearing ring (~1500 Hz)
    env_lo = env_lo - env_lo.mean()
    env_hi = env_hi - env_hi.mean()
    emag_lo = np.abs(np.fft.rfft(env_lo)) * 2.0 / n
    emag_hi = np.abs(np.fft.rfft(env_hi)) * 2.0 / n
    etot_lo = float(np.sqrt(np.sum(emag_lo ** 2))) + 1e-12
    etot_hi = float(np.sqrt(np.sum(emag_hi ** 2))) + 1e-12

    f_rot, f_bpfo = meta["f_rot_hz"], meta["f_bpfo_hz"]
    rot_energy = sum(_band_rms(emag_lo, ef, k * f_rot - 1.5, k * f_rot + 1.5) for k in (1, 2, 3))
    bpfo_energy = sum(_band_rms(emag_hi, ef, k * f_bpfo - 3.0, k * f_bpfo + 3.0) for k in (1, 2, 3))

    rms_v = float(np.sqrt(np.mean(av ** 2)))
    return dict(
        rms_vert_g=rms_v,
        rms_lat_g=float(np.sqrt(np.mean(al ** 2))),
        peak_vert_g=float(np.abs(av).max()),
        crest_vert=float(np.abs(av).max() / rms_v) if rms_v > 0 else np.nan,
        kurt_vert=_kurtosis(av),
        kurt_lat=_kurtosis(al),
        band_v_0p5_5=_band_rms(mag_v, freqs, 0.5, 5.0),
        band_v_5_20=_band_rms(mag_v, freqs, 5.0, 20.0),
        band_v_20_100=_band_rms(mag_v, freqs, 20.0, 100.0),
        band_v_100_500=_band_rms(mag_v, freqs, 100.0, 500.0),
        band_v_500_2500=_band_rms(mag_v, freqs, 500.0, 2500.0),
        band_l_0p5_5=_band_rms(mag_l, freqs, 0.5, 5.0),
        band_l_3_8=_band_rms(mag_l, freqs, 3.0, 8.0),
        ride_index_lat=_band_rms(mag_l, freqs, 0.5, 10.0),
        env_kurt=_kurtosis(env_hi),
        env_kurt_lo=_kurtosis(env_lo),
        env_rot_ratio=float(rot_energy / etot_lo),    # wheel-flat signature
        env_bpfo_ratio=float(bpfo_energy / etot_hi),  # bearing outer-race signature
        speed_kmh=float(meta["speed_kmh"]),
        f_rot_hz=float(f_rot),
        axlebox_temp_c=float(meta["axlebox_temp_c"]),
        axlebox_temp_rise_c=float(meta["axlebox_temp_c"] - meta["ambient_c"]),
        ambient_c=float(meta["ambient_c"]),
    )

## Cell 5 — fleet simulation loop

Walks the whole fleet day by day. Every event's features are extracted from a freshly
generated raw trace; the trace itself is kept only for a sampled subset (**stratified** —
faulty events are kept ~6× more often — so don't compute base rates from the raw store).

Runtime with the defaults is roughly half a minute. Scale `n_days` / `stops_per_day`
up once you're happy with the shape.

In [ ]:
# ── Cell 5: run the fleet simulation ───────────────────────────────────────────
import time

def run_simulation(cfg, doors_df, bogies_df, rng, verbose=True):
    door_rows, bogie_rows = [], []
    door_raw = {"key": [], "pos": [], "cur": [], "lock": []}
    bogie_raw = {"key": [], "acc_vert": [], "acc_lat": []}

    door_assets = doors_df.to_dict("records")
    bogie_assets = bogies_df.to_dict("records")
    lifetime = {d["asset_id"]: d["cycles0"] for d in door_assets}
    t_start = time.time()

    for day_idx, date in enumerate(DATES):
        weekend = date.dayofweek >= 5
        n_stops = int(cfg.stops_per_day * (0.6 if weekend else 1.0))
        n_runs = int(cfg.runs_per_day * (0.6 if weekend else 1.0))

        # ---------- doors ----------
        for door in door_assets:
            sev = severity_at(day_idx, door["onset_day"], door["ramp_days"], door["max_severity"])
            for k in range(n_stops):
                hour = 6.0 + 17.0 * (k + 0.5) / n_stops
                amb = ambient_c(day_idx, hour, cfg)
                t, p, i, lock, meta = simulate_door_cycle(door, sev, amb, rng, cfg)
                feats = extract_door_features(t, p, i, lock, meta, cfg.door_fs)
                lifetime[door["asset_id"]] += 1
                key = f'{door["asset_id"]}|{day_idx:03d}|{k:03d}'
                row = dict(
                    event_key=key, asset_id=door["asset_id"], asset_type="door",
                    carriage_id=door["carriage_id"], position=door["position"], route=door["route"],
                    timestamp=date + pd.Timedelta(hours=hour), day_idx=day_idx, hour=hour,
                    ambient_c=amb, cycles_lifetime=lifetime[door["asset_id"]],
                    **feats,
                    fault_type=door["fault_type"] if sev > 0 else "none",
                    onset_mode=door["onset_mode"] if sev > 0 else "none",
                    severity=sev,
                    label=int(sev > 0),
                    obstructed=meta["obstructed"],
                )
                door_rows.append(row)
                keep_p = cfg.raw_keep_fraction_door * (6.0 if sev > 0 else 1.0)
                if rng.random() < min(keep_p, 1.0):
                    door_raw["key"].append(key)
                    door_raw["pos"].append(p.astype(np.float32))
                    door_raw["cur"].append(i.astype(np.float32))
                    door_raw["lock"].append(lock.astype(np.int8))

        # ---------- bogies ----------
        for bogie in bogie_assets:
            sev = severity_at(day_idx, bogie["onset_day"], bogie["ramp_days"], bogie["max_severity"])
            for k in range(n_runs):
                hour = 6.0 + 17.0 * (k + 0.5) / n_runs
                amb = ambient_c(day_idx, hour, cfg)
                speed = float(np.clip(rng.normal(62.0, 14.0), 25.0, 100.0))
                t, av, al, meta = simulate_bogie_window(bogie, sev, speed, amb, rng, cfg)
                feats = extract_bogie_features(t, av, al, meta, cfg.bogie_fs)
                key = f'{bogie["asset_id"]}|{day_idx:03d}|{k:03d}'
                row = dict(
                    event_key=key, asset_id=bogie["asset_id"], asset_type="bogie",
                    carriage_id=bogie["carriage_id"], position=bogie["position"], route=bogie["route"],
                    timestamp=date + pd.Timedelta(hours=hour), day_idx=day_idx, hour=hour,
                    **feats,
                    fault_type=bogie["fault_type"] if sev > 0 else "none",
                    onset_mode=bogie["onset_mode"] if sev > 0 else "none",
                    severity=sev,
                    label=int(sev > 0),
                )
                bogie_rows.append(row)
                keep_p = cfg.raw_keep_fraction_bogie * (6.0 if sev > 0 else 1.0)
                if rng.random() < min(keep_p, 1.0):
                    bogie_raw["key"].append(key)
                    bogie_raw["acc_vert"].append(av.astype(np.float32))
                    bogie_raw["acc_lat"].append(al.astype(np.float32))

        if verbose and (day_idx % 10 == 0 or day_idx == cfg.n_days - 1):
            print(f"  day {day_idx + 1:>3}/{cfg.n_days}  "
                  f"door_events={len(door_rows):>7,}  bogie_windows={len(bogie_rows):>6,}  "
                  f"({time.time() - t_start:5.1f}s)")

    door_events = pd.DataFrame(door_rows)
    bogie_windows = pd.DataFrame(bogie_rows)
    for d in (door_raw, bogie_raw):
        for kk in list(d):
            if kk != "key":
                d[kk] = np.stack(d[kk]) if len(d[kk]) else np.zeros((0, 0))
    return door_events, bogie_windows, door_raw, bogie_raw


door_events, bogie_windows, door_raw, bogie_raw = run_simulation(CFG, doors_df, bogies_df, rng)

print(f"\ndoor_events   {door_events.shape}   anomalous {door_events.label.mean():.1%}")
print(f"bogie_windows {bogie_windows.shape}   anomalous {bogie_windows.label.mean():.1%}")
print(f"door_raw      {door_raw['pos'].shape}   bogie_raw {bogie_raw['acc_vert'].shape}")

## Cell 6 — sensor / data-quality fault injection

These are the *nuisance* anomalies. A stuck transducer or a drifting calibration also
produces a large reconstruction error, so an autoencoder will happily flag them as
"bogie fault" unless you handle them. They're injected **after** feature extraction,
on a per-asset, per-channel, time-bounded basis, and flagged in `dq_flag` / `dq_type`
so you can decide whether to exclude them, or keep them as a second class to evaluate.

In [ ]:
# ── Cell 6: data-quality faults ────────────────────────────────────────────────
DOOR_DQ_CHANNELS = ["peak_i_close_a", "mean_i_close_a", "door_gap_mm", "lock_delay_ms", "t_close_s"]
BOGIE_DQ_CHANNELS = ["rms_vert_g", "axlebox_temp_c", "rms_lat_g", "kurt_vert", "band_v_100_500"]


def inject_data_quality(df, channels, cfg, rng, label="asset"):
    df = df.copy()
    df["dq_flag"] = 0
    df["dq_type"] = "none"
    df["dq_channel"] = "none"

    assets = df["asset_id"].unique()
    k = max(len(SENSOR_FAULTS), int(round(cfg.p_sensor_fault * len(assets))))
    k = min(k, len(assets))
    chosen = list(rng.choice(assets, size=k, replace=False))
    dq_pool = list(SENSOR_FAULTS)
    rng.shuffle(dq_pool)
    episodes = []
    for j, a in enumerate(chosen):
        dq = dq_pool[j % len(dq_pool)]
        ch = str(rng.choice(channels))
        start = float(rng.integers(cfg.healthy_warmup_days, cfg.n_days - 2))
        dur = float(rng.uniform(3, 18))
        m = (df.asset_id == a) & (df.day_idx >= start) & (df.day_idx < start + dur)
        if not m.any():
            continue
        idx = df.index[m]
        vals = np.array(df.loc[idx, ch].to_numpy(dtype=float), copy=True)

        if dq == "stuck_at":
            vals[:] = vals[0]
        elif dq == "dropout":
            drop = rng.random(vals.size) < 0.45
            vals[drop] = np.nan
        elif dq == "clipping":
            cap = np.nanpercentile(vals, 55)
            vals = np.minimum(vals, cap)
        elif dq == "calibration_drift":
            ref = df.loc[df.asset_id == a, ch].astype(float)
            scale = float(np.nanstd(ref)) or 1.0
            vals = vals + np.linspace(0.0, rng.uniform(2.0, 4.5) * scale, vals.size)

        df.loc[idx, ch] = vals
        df.loc[idx, "dq_flag"] = 1
        df.loc[idx, "dq_type"] = dq
        df.loc[idx, "dq_channel"] = ch
        episodes.append(dict(asset_id=a, dq_type=dq, channel=ch,
                             start_day=start, end_day=start + dur, n_rows=int(m.sum())))
    if episodes:
        print(f"{label}: {len(episodes)} data-quality episodes")
        print(pd.DataFrame(episodes).to_string(index=False))
    return df, pd.DataFrame(episodes)


door_events, door_dq = inject_data_quality(door_events, DOOR_DQ_CHANNELS, CFG, rng, "doors")
bogie_windows, bogie_dq = inject_data_quality(bogie_windows, BOGIE_DQ_CHANNELS, CFG, rng, "bogies")

# convenience: one column that says what (if anything) is wrong with the row
def _status(r):
    if r.label == 1 and r.dq_flag == 1:
        return f"{r.fault_type}+{r.dq_type}"
    if r.label == 1:
        return r.fault_type
    if r.dq_flag == 1:
        return r.dq_type
    return "healthy"

for _df in (door_events, bogie_windows):
    _df["status"] = _df.apply(_status, axis=1)

print()
print(door_events.status.value_counts().to_string())
print()
print(bogie_windows.status.value_counts().to_string())

## Cell 7 — save to disk

Parquet if `pyarrow` is installed, otherwise CSV. Raw traces go to compressed `.npz`
keyed by `event_key`, so you can join a sequence back to its feature row.

In [ ]:
# ── Cell 7: persist ────────────────────────────────────────────────────────────
def save_table(df, name):
    try:
        path = os.path.join(OUT_DIR, f"{name}.parquet")
        df.to_parquet(path, index=False)
    except Exception:
        path = os.path.join(OUT_DIR, f"{name}.csv")
        df.to_csv(path, index=False)
    print(f"  {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {len(df):,} rows)")
    return path


print("Saving:")
save_table(door_events, "door_events")
save_table(bogie_windows, "bogie_windows")
save_table(doors_df, "assets_doors")
save_table(bogies_df, "assets_bogies")
if len(door_dq):
    save_table(door_dq, "dq_episodes_doors")
if len(bogie_dq):
    save_table(bogie_dq, "dq_episodes_bogies")

np.savez_compressed(
    os.path.join(OUT_DIR, "door_raw.npz"),
    event_key=np.array(door_raw["key"]), position=door_raw["pos"],
    current_a=door_raw["cur"], lock=door_raw["lock"], fs=np.array([CFG.door_fs]),
)
np.savez_compressed(
    os.path.join(OUT_DIR, "bogie_raw.npz"),
    event_key=np.array(bogie_raw["key"]), acc_vert=bogie_raw["acc_vert"],
    acc_lat=bogie_raw["acc_lat"], fs=np.array([CFG.bogie_fs]),
)
with open(os.path.join(OUT_DIR, "config.json"), "w") as fh:
    json.dump(asdict(CFG), fh, indent=2)
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:28s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:8.2f} MB")

## Cell 8 — sanity checks

Three quick looks: does a fault actually change the raw trace, does a gradual fault
actually drift, and does the bearing signature show up where it should.

In [ ]:
# ── Cell 8: sanity plots ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"     # categorical slots 1–3
GRID = dict(color="#d8d7d2", lw=0.6)
INK, INK2 = "#0b0b0b", "#52514e"


def _style(ax, title, xlabel, ylabel):
    ax.set_title(title, color=INK, fontsize=11, loc="left", pad=8)
    ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)
    ax.grid(True, axis="both", **GRID)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c9c8c3")
    ax.tick_params(colors=INK2, labelsize=8, length=3)


# --- 1. door motor current: healthy vs degraded actuator vs obstruction ---
demo = doors_df.iloc[0].to_dict()
demo_rng = np.random.default_rng(7)
t, p_h, i_h, l_h, m_h = simulate_door_cycle({**demo, "fault_type": "none"}, 0.0, 18.0, demo_rng, CFG)
_, _, i_d, _, _ = simulate_door_cycle({**demo, "fault_type": "actuator_degradation"}, 0.9, 18.0, demo_rng, CFG)
_, p_o, i_o, _, _ = simulate_door_cycle({**demo, "fault_type": "obstruction_prone"}, 0.9, 18.0, demo_rng, CFG,
                                        force_obstruction=True)

fig, ax = plt.subplots(figsize=(9, 3.4), dpi=120)
ax.plot(t, i_h, lw=1.6, color=C1, label="healthy")
ax.plot(t, i_d, lw=1.6, color=C2, label="actuator degradation (sev 0.9)")
ax.plot(t, i_o, lw=1.6, color=C3, label="obstruction + reopen")
_style(ax, "Door motor current — one open/close cycle", "time (s)", "current (A)")
ax.legend(frameon=False, fontsize=8, labelcolor=INK2, ncol=3, loc="upper right")
plt.tight_layout(); plt.show()

# --- 2. gradual degradation shows up as feature drift ---
grad = door_events[(door_events.label == 1) & (door_events.onset_mode == "gradual")]
if len(grad):
    aid = grad.asset_id.value_counts().index[0]
    one = (door_events[door_events.asset_id == aid].groupby("day_idx").peak_i_close_a.mean()
                       .rolling(3, min_periods=1, center=True).median())   # 3-day smoothing
    fleet = (door_events[door_events.asset_id.isin(doors_df.loc[doors_df.fault_type == "none", "asset_id"])]
             .groupby("day_idx").peak_i_close_a.median())
    onset = float(doors_df.loc[doors_df.asset_id == aid, "onset_day"].iloc[0])
    fig, ax = plt.subplots(figsize=(9, 3.2), dpi=120)
    ax.plot(fleet.index, fleet.values, lw=1.6, color=C1, label="healthy fleet median")
    ax.plot(one.index, one.values, lw=1.6, color=C2, label=f"{aid} (daily mean)")
    ax.axvline(onset, color=INK2, lw=1.0, ls=(0, (4, 3)))
    ax.annotate("fault onset", (onset, ax.get_ylim()[1]), xytext=(4, -10),
                textcoords="offset points", fontsize=8, color=INK2, va="top")
    _style(ax, "Peak closing current — the daily mean drifts upward after fault onset",
           "day", "peak current (A), 3-day median")
    ax.legend(frameon=False, fontsize=8, labelcolor=INK2, loc="upper left")
    plt.tight_layout(); plt.show()

# --- 3. bearing envelope signature ---
bb = bogie_windows[bogie_windows.fault_type == "bearing_bpfo"]
fig, ax = plt.subplots(figsize=(9, 3.2), dpi=120)
healthy = bogie_windows[bogie_windows.label == 0].groupby("day_idx").env_bpfo_ratio.median()
ax.plot(healthy.index, healthy.values, lw=1.6, color=C1, label="healthy fleet median")
if len(bb):
    aid = bb.asset_id.value_counts().index[0]
    ser = bogie_windows[bogie_windows.asset_id == aid].groupby("day_idx").env_bpfo_ratio.mean()
    ax.plot(ser.index, ser.values, lw=1.6, color=C2, label=f"{aid} (BPFO defect)")
_style(ax, "Envelope-spectrum BPFO ratio — bearing outer-race signature", "day", "BPFO energy / total envelope")
ax.legend(frameon=False, fontsize=8, labelcolor=INK2, loc="upper left")
plt.tight_layout(); plt.show()

## Cell 9 — autoencoder-ready splits, scaling and windowing

The one rule that matters: **train only on data you believe is healthy.** Here that's
the commissioning window (`healthy_warmup_days`), filtered to `label == 0` and
`dq_flag == 0`. Everything after it is validation/test with labels held out for scoring.

`fit_robust_scaler` uses median/IQR (no sklearn dependency); swap in
`sklearn.preprocessing.RobustScaler` if you'd rather.

In [ ]:
# ── Cell 9: splits, scaler, windowing ──────────────────────────────────────────
DOOR_FEATURES = ["t_open_s", "t_close_s", "peak_i_open_a", "peak_i_close_a", "mean_i_close_a",
                 "rms_i_close_a", "charge_as", "max_di_dt_a_s", "i_std_a", "n_reopen",
                 "lock_delay_ms", "door_gap_mm", "peak_v_norm_s", "cycle_incomplete", "ambient_c"]

BOGIE_FEATURES = ["rms_vert_g", "rms_lat_g", "peak_vert_g", "crest_vert", "kurt_vert", "kurt_lat",
                  "band_v_0p5_5", "band_v_5_20", "band_v_20_100", "band_v_100_500", "band_v_500_2500",
                  "band_l_0p5_5", "band_l_3_8", "ride_index_lat", "env_kurt", "env_kurt_lo", "env_rot_ratio",
                  "env_bpfo_ratio", "axlebox_temp_rise_c", "speed_kmh"]


def make_splits(df, features, cfg, val_days=12, drop_dq_from_train=True):
    """train = healthy commissioning window; val = next block; test = remainder."""
    train_end = cfg.healthy_warmup_days
    val_end = train_end + val_days
    is_clean = (df.label == 0) & ((df.dq_flag == 0) if drop_dq_from_train else True)
    train = df[(df.day_idx < train_end) & is_clean]
    val = df[(df.day_idx >= train_end) & (df.day_idx < val_end)]
    test = df[df.day_idx >= val_end]
    out = {}
    for nm, part in (("train", train), ("val", val), ("test", test)):
        out[nm] = dict(
            X=part[features].to_numpy(dtype=float),
            y=part["label"].to_numpy(),
            dq=part["dq_flag"].to_numpy(),
            sev=part["severity"].to_numpy(),
            fault=part["fault_type"].to_numpy(),
            meta=part[["event_key", "asset_id", "timestamp", "day_idx", "status"]].reset_index(drop=True),
        )
    return out


def fit_robust_scaler(X):
    med = np.nanmedian(X, axis=0)
    q75, q25 = np.nanpercentile(X, [75, 25], axis=0)
    iqr = np.where((q75 - q25) > 1e-9, q75 - q25, 1.0)
    return dict(center=med, scale=iqr)


def apply_scaler(X, sc, fill_nan=True):
    Z = (X - sc["center"]) / sc["scale"]
    if fill_nan:
        Z = np.where(np.isnan(Z), 0.0, Z)     # dropouts → 0 after centring; keep dq_flag as a feature if useful
    return Z


door_split = make_splits(door_events, DOOR_FEATURES, CFG)
bogie_split = make_splits(bogie_windows, BOGIE_FEATURES, CFG)
door_scaler = fit_robust_scaler(door_split["train"]["X"])
bogie_scaler = fit_robust_scaler(bogie_split["train"]["X"])

for nm, sp in (("door", door_split), ("bogie", bogie_split)):
    print(f"{nm}:")
    for part in ("train", "val", "test"):
        d = sp[part]
        print(f"  {part:5s} X={d['X'].shape}  anomaly rate={d['y'].mean():.1%}  dq rate={d['dq'].mean():.1%}")


def load_raw_sequences(npz_path, feature_df, channels, max_n=None):
    """Return (X[n, T, C], labels, meta) aligned to the feature table via event_key."""
    z = np.load(npz_path, allow_pickle=False)
    keys = z["event_key"]
    X = np.stack([z[c] for c in channels], axis=-1).astype(np.float32)
    lut = feature_df.set_index("event_key")
    keep = np.array([k in lut.index for k in keys])
    keys, X = keys[keep], X[keep]
    if max_n is not None and len(keys) > max_n:
        sel = np.random.default_rng(0).choice(len(keys), max_n, replace=False)
        keys, X = keys[sel], X[sel]
    meta = lut.loc[keys, ["asset_id", "day_idx", "label", "severity", "fault_type", "status"]].reset_index()
    return X, meta["label"].to_numpy(), meta


Xd, yd, md = load_raw_sequences(os.path.join(OUT_DIR, "door_raw.npz"), door_events,
                                ["position", "current_a"])
Xb, yb, mb = load_raw_sequences(os.path.join(OUT_DIR, "bogie_raw.npz"), bogie_windows,
                                ["acc_vert", "acc_lat"])
print(f"\nsequence tensors:  door {Xd.shape} (anomaly {yd.mean():.1%})   "
      f"bogie {Xb.shape} (anomaly {yb.mean():.1%})")
print("NOTE: the raw store is stratified (faulty kept ~6x) — use the feature tables for base rates.")


def rolling_asset_windows(df, features, win=10, step=1):
    """Per-asset rolling windows of daily-mean features → (n, win, F) for a sequence AE
    that models slow drift rather than single-event shape."""
    daily = (df.groupby(["asset_id", "day_idx"])[features].mean()
               .reset_index().sort_values(["asset_id", "day_idx"]))
    lab = df.groupby(["asset_id", "day_idx"]).label.max().reset_index()
    daily = daily.merge(lab, on=["asset_id", "day_idx"])
    Xs, ys, keys = [], [], []
    for aid, g in daily.groupby("asset_id"):
        A = g[features].to_numpy(dtype=float)
        L = g["label"].to_numpy()
        for s in range(0, len(g) - win + 1, step):
            Xs.append(A[s:s + win]); ys.append(int(L[s:s + win].max()))
            keys.append((aid, int(g.day_idx.iloc[s + win - 1])))
    return np.array(Xs), np.array(ys), pd.DataFrame(keys, columns=["asset_id", "end_day"])


Xw, yw, kw = rolling_asset_windows(door_events, DOOR_FEATURES, win=10)
print(f"door drift windows: {Xw.shape}  anomaly {yw.mean():.1%}")

## Data dictionary

### `door_events` — one row per open/close cycle

| column | meaning |
|---|---|
| `event_key` | join key to `door_raw.npz` |
| `asset_id`, `carriage_id`, `position`, `route` | identity |
| `timestamp`, `day_idx`, `hour` | when |
| `t_open_s`, `t_close_s` | measured travel times (0.05 ↔ 0.95 position crossings) |
| `peak_i_open_a`, `peak_i_close_a`, `mean_i_close_a`, `rms_i_close_a` | motor current statistics |
| `charge_as` | ∫ i dt over the cycle (A·s) — total work done |
| `max_di_dt_a_s`, `i_std_a` | current transient sharpness / roughness |
| `n_reopen` | direction reversals during closing (obstruction detections) |
| `lock_delay_ms`, `lock_unstable`, `cycle_incomplete` | lock switch behaviour; `cycle_incomplete` = the leaf never finished locking inside the capture window |
| `door_gap_mm` | leaf-to-frame gap when closed (alignment) |
| `cycles_lifetime`, `ambient_c` | operating context |
| `fault_type`, `severity`, `onset_mode`, `label` | **ground truth — evaluation only** |
| `dq_flag`, `dq_type`, `dq_channel` | injected data-quality fault |
| `status` | human-readable combination of the two |

### `bogie_windows` — one row per 2 s ride snapshot

| column | meaning |
|---|---|
| `rms_vert_g`, `rms_lat_g`, `peak_vert_g`, `crest_vert` | amplitude |
| `kurt_vert`, `kurt_lat`, `env_kurt`, `env_kurt_lo` | impulsiveness — `env_kurt` on the 0.8–2.4 kHz envelope (bearings), `env_kurt_lo` on the 150–600 Hz envelope (wheel flats) |
| `band_v_*`, `band_l_*` | RMS in frequency bands (Hz) — vertical / lateral |
| `ride_index_lat` | 0.5–10 Hz lateral RMS (ride comfort proxy) |
| `env_rot_ratio` | envelope energy at wheel-rotation harmonics → **wheel flat** |
| `env_bpfo_ratio` | envelope energy at ball-pass-outer-race harmonics → **bearing defect** |
| `axlebox_temp_c`, `axlebox_temp_rise_c` | thermal → **hot axlebox** |
| `speed_kmh`, `f_rot_hz`, `ambient_c` | operating context (keep these as AE inputs — the fault signatures are speed-dependent) |

### Suggested next steps

1. Fit a dense AE (e.g. 20 → 12 → 6 → 12 → 20, MSE) on `bogie_split['train']` scaled features.
2. Set the threshold at the 99th percentile of *training* reconstruction error, then score val/test.
3. Report PR-AUC against `label`, and separately against `dq_flag`, to see how much of your
   detection is really just data-quality faults.
4. Plot per-asset reconstruction error vs `day_idx` — for gradual faults the useful metric is
   *lead time* (days between first alarm and `severity == max`), not point accuracy.

In [ ]:
print(door_events)
short = door_events.iloc[:1000,:]
plt.scatter(short["timestamp"],short["peak_v_norm_s"])
plt.xticks(rotation=90)

In [ ]:
door_events.to_pickle(r"C:\Users\keith\PycharmProjects\NebulaX\door_events.pkl")